# Feature-Based Model: Random Forest — Full NeuroKit2 Processing on All 12 Leads

**Dataset:** Chapman-Shaoxing (AFib + Normal Sinus Rhythm subset)  
**Task:** Binary classification — AFib vs Normal  
**Model:** Random Forest on morphological features  
**Approach:** Full NeuroKit2 `ecg_process()` run independently on EACH of the 12 leads  
**Setup:** 5-fold stratified CV by patient IDs (same split, seed=42)

### Features (132 total):
- **Per lead (12 leads × 11 features = 132):** P-wave ratio, P-wave amplitude, P-wave duration, QRS duration, R-peak amplitude, T-wave amplitude, PR interval, QT interval, Mean RR, Heart rate, Beat count

**Note:** This approach runs NeuroKit2's full ECG processing pipeline independently on each lead. Estimated processing time: ~4-5 hours.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install wfdb neurokit2 -q
print('Dependencies installed.')

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.9/688.9 kB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 152.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
Dependencies installed.


## 1. Imports

In [2]:
import os
import numpy as np
import neurokit2 as nk
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, matthews_corrcoef,
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    brier_score_loss
)
from sklearn.calibration import calibration_curve
import time
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
import wfdb

print(f"NeuroKit2 version: {nk.__version__}")
print("Imports done.")

NeuroKit2 version: 0.2.13
Imports done.


## 2. Configuration

In [3]:
class Config:
    afib_dir = ""
    normal_dir = ""

    fs = 500
    num_channels = 12
    recording_seconds = 10
    recording_samples = fs * recording_seconds
    lead_names = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

    n_estimators = 500
    class_weight = "balanced"
    n_jobs = -1

    n_folds = 5
    random_seed = 42

    pdf_dir = "/content/drive/Othercomputers/Min_PC/Skrivebord/Afib-Master/plots_pdf_rf_morph_all_leads"
    model_save_dir = "/content/drive/Othercomputers/Min_PC/Skrivebord/Afib-Master/Best model"

config = Config()
os.makedirs(config.pdf_dir, exist_ok=True)
os.makedirs(config.model_save_dir, exist_ok=True)
print("Config ready.")

Config ready.


## 3. Unzip Data

In [4]:
if not os.path.exists("/content/subset_data"):
    print("Unzipping subset...")
    !unzip -q "/content/drive/Othercomputers/Min_PC/Skrivebord/Afib-Master/data/afib_normal_subset_ecg_arrhythmia.zip" -d /content/subset_data/

for base in ["/content/subset_data", "/content/subset_data/afib_normal_subset",
             "/content/subset_data/afib_normal_subset_ecg_arrhythmia"]:
    if os.path.exists(os.path.join(base, "afib")):
        config.afib_dir = os.path.join(base, "afib")
        config.normal_dir = os.path.join(base, "normal")
        break

print(f"AFib: {len([f for f in os.listdir(config.afib_dir) if f.endswith('.hea')])} records")
print(f"Normal: {len([f for f in os.listdir(config.normal_dir) if f.endswith('.hea')])} records")

Unzipping subset...
AFib: 1780 records
Normal: 8125 records


## 4. Discover Records

In [5]:
afib_files = sorted([f[:-4] for f in os.listdir(config.afib_dir) if f.endswith('.hea')])
normal_files = sorted([f[:-4] for f in os.listdir(config.normal_dir) if f.endswith('.hea')])

all_records = []
for name in afib_files:
    all_records.append({"path": os.path.join(config.afib_dir, name), "name": name, "label": 1})
for name in normal_files:
    all_records.append({"path": os.path.join(config.normal_dir, name), "name": name, "label": 0})

print(f"AFib: {len(afib_files)}, Normal: {len(normal_files)}, Total: {len(all_records)}")

AFib: 1780, Normal: 8125, Total: 9905


## 5. Feature Extraction — Full Processing on Each Lead

NeuroKit2's `ecg_process()` is run independently on each of the 12 leads, extracting the same 11 morphological features per lead.

In [6]:
def extract_features_single_lead(signal_1lead, fs):
    """Extract 11 morphological features from a single ECG lead using NeuroKit2."""
    try:
        signals, info = nk.ecg_process(signal_1lead, sampling_rate=fs)

        r_peaks = info["ECG_R_Peaks"]
        n_beats = len(r_peaks)

        if n_beats < 3:
            return None

        # R-peak amplitude
        r_amplitudes = signals["ECG_Clean"].values[r_peaks]
        mean_r_amplitude = np.mean(r_amplitudes)

        # RR intervals
        rr_intervals = np.diff(r_peaks) / fs
        mean_rr = np.mean(rr_intervals)
        heart_rate = 60.0 / mean_rr if mean_rr > 0 else 0

        # P-wave features
        p_peak_indices = np.where(signals["ECG_P_Peaks"].values == 1)[0]
        p_onset_indices = np.where(signals["ECG_P_Onsets"].values == 1)[0]
        p_offset_indices = np.where(signals["ECG_P_Offsets"].values == 1)[0]

        # P-wave detection ratio
        p_detected = 0
        for rp in r_peaks:
            ws = rp - int(0.3 * fs)
            we = rp - int(0.05 * fs)
            if ws >= 0 and np.any((p_peak_indices >= ws) & (p_peak_indices <= we)):
                p_detected += 1
        p_wave_ratio = p_detected / n_beats

        # P-wave amplitude
        mean_p_amplitude = np.mean(signals["ECG_Clean"].values[p_peak_indices]) if len(p_peak_indices) > 0 else 0.0

        # P-wave duration
        p_durs = []
        for onset in p_onset_indices:
            offs = p_offset_indices[p_offset_indices > onset]
            if len(offs) > 0:
                d = (offs[0] - onset) / fs * 1000
                if 20 < d < 200:
                    p_durs.append(d)
        mean_p_duration = np.mean(p_durs) if p_durs else 0.0

        # QRS duration
        r_onset_indices = np.where(signals["ECG_R_Onsets"].values == 1)[0]
        r_offset_indices = np.where(signals["ECG_R_Offsets"].values == 1)[0]
        qrs_durs = []
        for onset in r_onset_indices:
            offs = r_offset_indices[r_offset_indices > onset]
            if len(offs) > 0:
                d = (offs[0] - onset) / fs * 1000
                if 40 < d < 200:
                    qrs_durs.append(d)
        mean_qrs_duration = np.mean(qrs_durs) if qrs_durs else 0.0

        # T-wave amplitude
        t_peak_indices = np.where(signals["ECG_T_Peaks"].values == 1)[0]
        mean_t_amplitude = np.mean(signals["ECG_Clean"].values[t_peak_indices]) if len(t_peak_indices) > 0 else 0.0

        # PR interval
        pr_ints = []
        for rp in r_peaks:
            ws = rp - int(0.3 * fs)
            if ws < 0: continue
            p_before = p_onset_indices[(p_onset_indices >= ws) & (p_onset_indices < rp)]
            if len(p_before) > 0:
                pr = (rp - p_before[-1]) / fs * 1000
                if 80 < pr < 350:
                    pr_ints.append(pr)
        mean_pr = np.mean(pr_ints) if pr_ints else 0.0

        # QT interval
        t_offset_indices = np.where(signals["ECG_T_Offsets"].values == 1)[0]
        qt_ints = []
        for onset in r_onset_indices:
            t_offs = t_offset_indices[t_offset_indices > onset]
            if len(t_offs) > 0:
                qt = (t_offs[0] - onset) / fs * 1000
                if 200 < qt < 600:
                    qt_ints.append(qt)
        mean_qt = np.mean(qt_ints) if qt_ints else 0.0

        features = [p_wave_ratio, mean_p_amplitude, mean_p_duration,
                    mean_qrs_duration, mean_r_amplitude, mean_t_amplitude,
                    mean_pr, mean_qt, mean_rr, heart_rate, n_beats]

        return features

    except Exception:
        return None


def extract_features_all_leads(signal, fs, config):
    """Run full NeuroKit2 processing on each of the 12 leads independently."""
    all_features = []

    for ch in range(config.num_channels):
        lead_features = extract_features_single_lead(signal[:, ch], fs)

        if lead_features is None:
            # If a lead fails, fill with zeros
            all_features.extend([0.0] * 11)
        else:
            all_features.extend(lead_features)

    features = np.array(all_features, dtype=np.float32)

    if np.any(np.isnan(features)) or np.any(np.isinf(features)):
        features = np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)

    return features


# Build feature names
feature_per_lead = ['P_Wave_Ratio', 'P_Wave_Amplitude', 'P_Wave_Duration_ms',
                    'QRS_Duration_ms', 'R_Peak_Amplitude', 'T_Wave_Amplitude',
                    'PR_Interval_ms', 'QT_Interval_ms', 'Mean_RR_s', 'Heart_Rate_bpm', 'Beat_Count']

feature_names = []
for lead in config.lead_names:
    for feat in feature_per_lead:
        feature_names.append(f"{lead}_{feat}")

print(f"Total features: {len(feature_names)} ({config.num_channels} leads x {len(feature_per_lead)} features)")

Total features: 132 (12 leads x 11 features)


## 6. Extract Features from All Records

In [7]:
print(f"Extracting features (full processing, all 12 leads) from {len(all_records)} records...")
print("Estimated time: 4-5 hours\n")
start_time = time.time()

patient_features = {}
skipped = 0

for i, rec in enumerate(all_records):
    if (i + 1) % 200 == 0:
        elapsed = time.time() - start_time
        pct = (i + 1) / len(all_records) * 100
        est_total = elapsed / (i + 1) * len(all_records)
        est_remaining = est_total - elapsed
        print(f"  Processed {i+1}/{len(all_records)} ({pct:.0f}%) [{elapsed/60:.1f} min elapsed, ~{est_remaining/60:.0f} min remaining]")

    try:
        record = wfdb.rdrecord(rec["path"])
        signal = record.p_signal

        if signal.shape[1] < config.num_channels or np.any(np.isnan(signal)):
            skipped += 1
            continue

        if len(signal) < config.recording_samples:
            signal = np.vstack([signal, np.zeros((config.recording_samples - len(signal), config.num_channels))])
        elif len(signal) > config.recording_samples:
            signal = signal[:config.recording_samples, :]

        features = extract_features_all_leads(signal, config.fs, config)
        patient_features[rec["name"]] = {"features": features, "label": rec["label"]}

    except Exception:
        skipped += 1

load_time = time.time() - start_time
print(f"\nExtraction complete in {load_time/60:.1f} minutes")
print(f"Patients loaded: {len(patient_features)} (skipped: {skipped})")

afib_patients = sum(1 for p in patient_features.values() if p["label"] == 1)
normal_patients = sum(1 for p in patient_features.values() if p["label"] == 0)
print(f"  AFib: {afib_patients}, Normal: {normal_patients}")

Extracting features (full processing, all 12 leads) from 9905 records...
Estimated time: 4-5 hours

  Processed 200/9905 (2%) [5.8 min elapsed, ~283 min remaining]
  Processed 400/9905 (4%) [11.7 min elapsed, ~277 min remaining]
  Processed 600/9905 (6%) [17.5 min elapsed, ~272 min remaining]
  Processed 800/9905 (8%) [23.4 min elapsed, ~267 min remaining]
  Processed 1000/9905 (10%) [29.3 min elapsed, ~261 min remaining]
  Processed 1200/9905 (12%) [35.1 min elapsed, ~255 min remaining]
  Processed 1400/9905 (14%) [41.0 min elapsed, ~249 min remaining]
  Processed 1600/9905 (16%) [46.8 min elapsed, ~243 min remaining]
  Processed 1800/9905 (18%) [52.6 min elapsed, ~237 min remaining]
  Processed 2000/9905 (20%) [58.0 min elapsed, ~229 min remaining]
  Processed 2200/9905 (22%) [63.4 min elapsed, ~222 min remaining]
  Processed 2400/9905 (24%) [68.8 min elapsed, ~215 min remaining]
  Processed 2600/9905 (26%) [74.2 min elapsed, ~208 min remaining]
  Processed 2800/9905 (28%) [79.5 min 

## 7. 5-Fold Stratified Split (Same seed=42)

In [8]:
np.random.seed(config.random_seed)
patient_names = np.array(sorted(patient_features.keys()))
patient_labels = np.array([patient_features[p]["label"] for p in patient_names])

skf = StratifiedKFold(n_splits=config.n_folds, shuffle=True, random_state=config.random_seed)
folds = []

print("=" * 80)
print("5-FOLD SPLIT (same seed=42)")
print("=" * 80)
print(f"{'Fold':<6} {'Train':<10} {'Val':<10} {'Val AFib':<12} {'Val Normal':<12} {'AFib %'}")
print("-" * 60)

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(patient_names, patient_labels)):
    train_p = patient_names[train_idx].tolist()
    val_p = patient_names[val_idx].tolist()
    folds.append({"train": train_p, "val": val_p})
    va = sum(1 for p in val_p if patient_features[p]["label"] == 1)
    vn = sum(1 for p in val_p if patient_features[p]["label"] == 0)
    print(f"{fold_idx+1:<6} {len(train_p):<10} {len(val_p):<10} {va:<12} {vn:<12} {va/(va+vn)*100:.1f}%")

5-FOLD SPLIT (same seed=42)
Fold   Train      Val        Val AFib     Val Normal   AFib %
------------------------------------------------------------
1      7911       1978       356          1622         18.0%
2      7911       1978       356          1622         18.0%
3      7911       1978       356          1622         18.0%
4      7911       1978       356          1622         18.0%
5      7912       1977       356          1621         18.0%


## 8. Helper Functions

In [9]:
def merge_patients(patient_list):
    X = np.array([patient_features[p]["features"] for p in patient_list], dtype=np.float32)
    y = np.array([patient_features[p]["label"] for p in patient_list], dtype=np.int64)
    return X, y

def compute_all_metrics(targets, preds, probs):
    cm = confusion_matrix(targets, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    brier = brier_score_loss(targets, probs)
    n_bins = 10
    boundaries = np.linspace(0, 1, n_bins + 1)
    ece, mce = 0.0, 0.0
    for i in range(n_bins):
        mask = (probs >= boundaries[i]) & (probs < boundaries[i+1]) if i < n_bins-1 else (probs >= boundaries[i]) & (probs <= boundaries[i+1])
        if mask.sum() > 0:
            err = abs(targets[mask].mean() - probs[mask].mean())
            ece += (mask.sum() / len(targets)) * err
            mce = max(mce, err)
    return {"tn": tn, "fp": fp, "fn": fn, "tp": tp,
            "accuracy": accuracy_score(targets, preds),
            "precision": precision_score(targets, preds, zero_division=0),
            "recall": recall_score(targets, preds, zero_division=0),
            "f1": f1_score(targets, preds, zero_division=0),
            "mcc": matthews_corrcoef(targets, preds),
            "auroc": roc_auc_score(targets, probs),
            "auprc": average_precision_score(targets, probs),
            "brier": brier, "ece": ece, "mce": mce, "cm": cm}

print("Functions defined.")

Functions defined.


## 9. Run 5-Fold CV

In [17]:
all_fold_metrics = []
all_fold_targets = []
all_fold_preds = []
all_fold_probs = []
all_fold_models = []

total_start = time.time()

for fold_idx, fold in enumerate(folds):
    print(f"\n{'='*60}")
    print(f"FOLD {fold_idx+1}/{config.n_folds}")
    print(f"{'='*60}")

    X_train, y_train = merge_patients(fold["train"])
    X_val, y_val = merge_patients(fold["val"])
    print(f"  Train: {len(y_train)} (AFib: {np.sum(y_train==1)}, Normal: {np.sum(y_train==0)})")
    print(f"  Val:   {len(y_val)} (AFib: {np.sum(y_val==1)}, Normal: {np.sum(y_val==0)})")
    print(f"  Features: {X_train.shape[1]}")

    rf = RandomForestClassifier(
        n_estimators=config.n_estimators,
        class_weight=config.class_weight,
        random_state=config.random_seed,
        n_jobs=config.n_jobs
    )
    rf.fit(X_train, y_train)

    preds = rf.predict(X_val)
    probs = rf.predict_proba(X_val)[:, 1]
    metrics = compute_all_metrics(y_val, preds, probs)

    all_fold_metrics.append(metrics)
    all_fold_targets.append(y_val)
    all_fold_preds.append(preds)
    all_fold_probs.append(probs)
    all_fold_models.append(rf)

    print(f"  Acc={metrics['accuracy']:.4f} F1={metrics['f1']:.4f} Recall={metrics['recall']:.4f} AUROC={metrics['auroc']:.4f}")
    print(f"  TN={metrics['tn']} FP={metrics['fp']} FN={metrics['fn']} TP={metrics['tp']}")
    print(f"  ECE={metrics['ece']:.4f} MCE={metrics['mce']:.4f} Brier={metrics['brier']:.4f}")

total_time = time.time() - total_start
print(f"\nTotal CV time: {total_time:.1f} seconds")


FOLD 1/5
  Train: 7911 (AFib: 1424, Normal: 6487)
  Val:   1978 (AFib: 356, Normal: 1622)
  Features: 132
  Acc=0.9687 F1=0.9069 Recall=0.8483 AUROC=0.9966
  TN=1614 FP=8 FN=54 TP=302
  ECE=0.0580 MCE=0.3787 Brier=0.0260

FOLD 2/5
  Train: 7911 (AFib: 1424, Normal: 6487)
  Val:   1978 (AFib: 356, Normal: 1622)
  Features: 132
  Acc=0.9737 F1=0.9242 Recall=0.8904 AUROC=0.9955
  TN=1609 FP=13 FN=39 TP=317
  ECE=0.0563 MCE=0.3296 Brier=0.0259

FOLD 3/5
  Train: 7911 (AFib: 1424, Normal: 6487)
  Val:   1978 (AFib: 356, Normal: 1622)
  Features: 132
  Acc=0.9596 F1=0.8773 Recall=0.8034 AUROC=0.9945
  TN=1612 FP=10 FN=70 TP=286
  ECE=0.0526 MCE=0.4546 Brier=0.0298

FOLD 4/5
  Train: 7911 (AFib: 1424, Normal: 6487)
  Val:   1978 (AFib: 356, Normal: 1622)
  Features: 132
  Acc=0.9661 F1=0.8989 Recall=0.8371 AUROC=0.9953
  TN=1613 FP=9 FN=58 TP=298
  ECE=0.0543 MCE=0.3247 Brier=0.0290

FOLD 5/5
  Train: 7912 (AFib: 1424, Normal: 6488)
  Val:   1977 (AFib: 356, Normal: 1621)
  Features: 132
  A

## 10. Results Summary

In [ ]:
metric_names = ["accuracy", "precision", "recall", "f1", "mcc", "auroc", "auprc", "brier", "ece", "mce"]

print("=" * 100)
print("5-FOLD CV RESULTS — RF Full Processing All 12 Leads (NeuroKit2)")
print("=" * 100)
header = f"{'Metric':<12}"
for i in range(len(folds)): header += f"{'Fold '+str(i+1):<14}"
header += f"{'Mean':<14}{'Std':<14}"
print(header)
print("-" * 100)

for metric in metric_names + ["tn", "fp", "fn", "tp"]:
    values = [m[metric] for m in all_fold_metrics]
    row = f"{metric.upper():<12}"
    for v in values:
        row += f"{int(v):<14}" if metric in ["tn","fp","fn","tp"] else f"{v:<14.4f}"
    mean_v, std_v = np.mean(values), np.std(values)
    row += f"{mean_v:<14.0f}{std_v:<14.0f}" if metric in ["tn","fp","fn","tp"] else f"{mean_v:<14.4f}{std_v:<14.4f}"
    print(row)
print("=" * 100)

## 11. Feature Importance (Top 20)

In [ ]:
mean_imp = np.mean([rf.feature_importances_ for rf in all_fold_models], axis=0)
sorted_idx = np.argsort(mean_imp)[::-1]
top_n = 20

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(top_n), mean_imp[sorted_idx[:top_n]][::-1], color='#4472C4', edgecolor='black', alpha=0.8)
ax.set_yticks(range(top_n))
ax.set_yticklabels([feature_names[i] for i in sorted_idx[:top_n]][::-1], fontsize=10)
ax.set_xlabel('Mean Feature Importance', fontsize=12)
ax.set_title('Top 20 Features — RF Full Processing All 12 Leads', fontsize=13, fontweight='bold')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(config.pdf_dir, "rf_morph_all_feature_importance.pdf"), format="pdf", bbox_inches="tight", dpi=300)
plt.show()

print("\nTop 20 features:")
for rank, idx in enumerate(sorted_idx[:20]):
    print(f"  {rank+1:2d}. {feature_names[idx]:<30s} {mean_imp[idx]:.4f}")

## 12. Plots

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 5, figsize=(25, 4.5))
for i, m in enumerate(all_fold_metrics):
    ax = axes[i]
    sns.heatmap(m["cm"], annot=True, fmt="d", cmap="Greens",
                xticklabels=["Normal","AFib"], yticklabels=["Normal","AFib"], ax=ax, annot_kws={"size":12})
    ax.set_title(f"Fold {i+1}\nAcc={m['accuracy']:.3f} F1={m['f1']:.3f}", fontsize=11, fontweight='bold')
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual" if i==0 else "")
plt.suptitle("Confusion Matrices — RF Full Processing All Leads", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(config.pdf_dir, "rf_morph_all_confusion_matrices.pdf"), format="pdf", bbox_inches="tight", dpi=300)
plt.show()

# ROC Curves
fig, ax = plt.subplots(figsize=(8, 7))
mean_fpr = np.linspace(0, 1, 100); tprs = []
for i in range(len(folds)):
    fpr, tpr, _ = roc_curve(all_fold_targets[i], all_fold_probs[i])
    ax.plot(fpr, tpr, alpha=0.3, linewidth=1.5, label=f'Fold {i+1} (AUROC={all_fold_metrics[i]["auroc"]:.3f})')
    t = np.interp(mean_fpr, fpr, tpr); t[0] = 0.0; tprs.append(t)
mean_tpr = np.mean(tprs, axis=0); mean_tpr[-1] = 1.0
m_auroc = np.mean([m["auroc"] for m in all_fold_metrics])
ax.plot(mean_fpr, mean_tpr, color='green', linewidth=2.5, label=f'Mean (AUROC={m_auroc:.3f})')
ax.plot([0,1],[0,1],'k--',alpha=0.5)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC — RF Full Processing All Leads', fontsize=14, fontweight='bold')
ax.legend(loc='lower right'); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(config.pdf_dir, "rf_morph_all_roc_curves.pdf"), format="pdf", bbox_inches="tight", dpi=300)
plt.show()

# Calibration Plots
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
for i in range(len(folds)):
    row, col = i // 3, i % 3; ax = axes[row][col]
    frac, pred = calibration_curve(all_fold_targets[i], all_fold_probs[i], n_bins=10, strategy='uniform')
    ax.plot(pred, frac, 's-', color='green', linewidth=2, markersize=6, label='Model')
    ax.plot([0,1],[0,1],'k--',alpha=0.5,label='Perfect')
    ax2 = ax.twinx(); ax2.hist(all_fold_probs[i], bins=20, range=(0,1), alpha=0.15, color='green')
    m = all_fold_metrics[i]
    ax.set_title(f"Fold {i+1}\nECE={m['ece']:.4f} MCE={m['mce']:.4f}", fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted Prob'); ax.set_ylabel('Fraction Positive')
    ax.legend(loc='upper left',fontsize=9); ax.set_xlim([0,1]); ax.set_ylim([0,1]); ax.grid(True,alpha=0.3)
ax = axes[1][2]
all_t = np.concatenate(all_fold_targets); all_p = np.concatenate(all_fold_probs)
frac, pred = calibration_curve(all_t, all_p, n_bins=10, strategy='uniform')
ax.plot(pred, frac, 's-', color='green', linewidth=2.5, markersize=8, label='Combined')
ax.plot([0,1],[0,1],'k--',alpha=0.5)
ax.set_title(f"Combined\nMean ECE={np.mean([m['ece'] for m in all_fold_metrics]):.4f} MCE={np.mean([m['mce'] for m in all_fold_metrics]):.4f}", fontsize=11, fontweight='bold')
ax.legend(loc='upper left',fontsize=9); ax.set_xlim([0,1]); ax.set_ylim([0,1]); ax.grid(True,alpha=0.3)
plt.suptitle("Calibration — RF Full Processing All Leads", fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(config.pdf_dir, "rf_morph_all_calibration_plots.pdf"), format="pdf", bbox_inches="tight", dpi=300)
plt.show()

# Probability Distributions
fig, axes = plt.subplots(1, 5, figsize=(25, 4.5))
for i in range(len(folds)):
    ax = axes[i]
    ax.hist(all_fold_probs[i][all_fold_targets[i]==0], bins=50, range=(0,1), alpha=0.6, color='green', label='Normal', density=True)
    ax.hist(all_fold_probs[i][all_fold_targets[i]==1], bins=50, range=(0,1), alpha=0.6, color='red', label='AFib', density=True)
    ax.axvline(x=0.5, color='black', linestyle='--', alpha=0.5)
    ax.set_title(f"Fold {i+1}", fontweight='bold'); ax.set_xlabel("P(AFib)")
    if i==0: ax.set_ylabel("Density"); ax.legend(fontsize=9)
plt.suptitle("Probability Distributions — RF Full Processing All Leads", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(config.pdf_dir, "rf_morph_all_probability_distributions.pdf"), format="pdf", bbox_inches="tight", dpi=300)
plt.show()

## 13. Comparison: All Approaches

In [14]:
print("=" * 95)
print("COMPARISON: All Models on Chapman-Shaoxing")
print("=" * 95)
print(f"\n{'Metric':<12} {'KanResWideX':<15} {'CNN-BiLSTM':<15} {'RF (LeadII)':<15} {'RF (12L-Opt)':<15} {'RF (12L-Full)':<15}")
print("-" * 87)

kanres = {"accuracy":0.9952,"precision":0.9823,"recall":0.9916,"f1":0.9869,"mcc":0.9840,
          "auroc":0.9991,"auprc":0.9941,"brier":0.0042,"ece":0.0038,"mce":0.5373}
bilstm = {"accuracy":0.9903,"precision":0.9577,"recall":0.9899,"f1":0.9735,"mcc":0.9678,
          "auroc":0.9983,"auprc":0.9895,"brier":0.0082,"ece":0.0072,"mce":0.5204}
rf_l2 = {"accuracy":0.9383,"precision":0.8742,"recall":0.7676,"f1":0.8173,"mcc":0.7828,
         "auroc":0.9749,"auprc":0.9388,"brier":0.0466,"ece":0.0206,"mce":0.1817}
rf_12opt = {"accuracy":0.9474,"precision":0.9128,"recall":0.7822,"f1":0.8423,"mcc":0.8146,
            "auroc":0.9858,"auprc":0.9388,"brier":0.0389,"ece":0.0386,"mce":0.2476}

for metric in ["accuracy","precision","recall","f1","mcc","auroc","auprc","brier","ece","mce"]:
    rf_full = np.mean([m[metric] for m in all_fold_metrics])
    print(f"{metric.upper():<12} {kanres[metric]:<15.4f} {bilstm[metric]:<15.4f} {rf_l2[metric]:<15.4f} {rf_12opt[metric]:<15.4f} {rf_full:<15.4f}")

COMPARISON: All Models on Chapman-Shaoxing

Metric       KanResWideX     CNN-BiLSTM      RF (LeadII)     RF (12L-Opt)    RF (12L-Full)  
---------------------------------------------------------------------------------------
ACCURACY     0.9952          0.9903          0.9383          0.9474          0.9670         
PRECISION    0.9823          0.9577          0.8742          0.9128          0.9685         
RECALL       0.9916          0.9899          0.7676          0.7822          0.8444         
F1           0.9869          0.9735          0.8173          0.8423          0.9019         
MCC          0.9840          0.9678          0.7828          0.8146          0.8854         
AUROC        0.9991          0.9983          0.9749          0.9858          0.9956         
AUPRC        0.9941          0.9895          0.9388          0.9388          0.9772         
BRIER        0.0042          0.0082          0.0466          0.0389          0.0275         
ECE          0.0038          0.

## 14. Final Summary

In [15]:
print("=" * 70)
print("FINAL SUMMARY — RF Full NeuroKit2 Processing All 12 Leads")
print("=" * 70)
for m in ["accuracy","precision","recall","f1","mcc","auroc","auprc","brier","ece","mce"]:
    mean_v = np.mean([f[m] for f in all_fold_metrics])
    std_v = np.std([f[m] for f in all_fold_metrics])
    print(f"  {m.upper():<15} {mean_v:.4f} +/- {std_v:.4f}")
print(f"\nFeatures: {len(feature_names)} ({config.num_channels} leads x {len(feature_per_lead)} per lead)")
print(f"Patients: {len(patient_features)} (AFib: {afib_patients}, Normal: {normal_patients})")

FINAL SUMMARY — RF Full NeuroKit2 Processing All 12 Leads
  ACCURACY        0.9670 +/- 0.0046
  PRECISION       0.9685 +/- 0.0047
  RECALL          0.8444 +/- 0.0278
  F1              0.9019 +/- 0.0151
  MCC             0.8854 +/- 0.0163
  AUROC           0.9956 +/- 0.0007
  AUPRC           0.9772 +/- 0.0063
  BRIER           0.0275 +/- 0.0016
  ECE             0.0559 +/- 0.0022
  MCE             0.3932 +/- 0.0631

Features: 132 (12 leads x 11 per lead)
Patients: 9889 (AFib: 1780, Normal: 8109)


## 15. Save Best Model

In [16]:
best_idx = np.argmax([m["f1"] for m in all_fold_metrics])
print(f"Best fold: {best_idx+1} (F1={all_fold_metrics[best_idx]['f1']:.4f})")

save_path = os.path.join(config.model_save_dir, "best_rf_morph_all_leads_model.pkl")
with open(save_path, 'wb') as f:
    pickle.dump(all_fold_models[best_idx], f)
print(f"Model saved: {save_path}")

feat_path = os.path.join(config.model_save_dir, "rf_morph_all_leads_feature_names.pkl")
with open(feat_path, 'wb') as f:
    pickle.dump(feature_names, f)
print(f"Feature names saved: {feat_path}")

print(f"\nPDFs in: {config.pdf_dir}")
for fn in sorted(os.listdir(config.pdf_dir)):
    if fn.endswith('.pdf'): print(f"  {fn}")

Best fold: 2 (F1=0.9242)
Model saved: /content/drive/Othercomputers/Min_PC/Skrivebord/Afib-Master/Best model/best_rf_morph_all_leads_model.pkl
Feature names saved: /content/drive/Othercomputers/Min_PC/Skrivebord/Afib-Master/Best model/rf_morph_all_leads_feature_names.pkl

PDFs in: /content/drive/Othercomputers/Min_PC/Skrivebord/Afib-Master/plots_pdf_rf_morph_all_leads
  rf_morph_all_calibration_plots.pdf
  rf_morph_all_confusion_matrices.pdf
  rf_morph_all_feature_importance.pdf
  rf_morph_all_probability_distributions.pdf
  rf_morph_all_roc_curves.pdf
